In [1]:
from data_meta_map.task2vec import task2vec
from data_meta_map.models import get_model
from data_meta_map import datasets
from data_meta_map.task2vec import plot_distance_matrix
from data_meta_map.task2vec import Task2Vec
from data_meta_map.task2vec.task_similarity import cosine

import numpy as np

/home/machenike/bmml/DataMetaMap/src/data_meta_map/wasserstein_embedder.py:8: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
import yaml
from collections import defaultdict

def get_pretrained_results(dataset_names, path_to_logs):
    pretrain2downstream_results = defaultdict(dict)
    for pretrain_name in dataset_names:
        for downstream_name in dataset_names:
            if downstream_name == 'imagenet':
                continue
            if downstream_name == pretrain_name:
                continue
            with open(f'{path_to_logs}/{pretrain_name}-{downstream_name}.yaml', 'r') as f:
                test_acc = yaml.safe_load(f)['task']['test_acc']
                pretrain2downstream_results[pretrain_name][downstream_name] = test_acc
    return pretrain2downstream_results

def get_embedder_results(dataset_names, path_to_pretrained_logs, embedder_func, similarity_func):
    embeddings = []

    dataset_list = [datasets.__dict__[name](root='../../data')[0] for name in dataset_names]
    for name, dataset in zip(dataset_names, dataset_list):
        embeddings.append(embedder_func(dataset))

    def find_closest(dataset_names, embeddings, similarity_metric):
        res = dict()
        for dataset_name, embed in zip(dataset_names, embeddings):
            dists = {name:similarity_metric(embed, other_embed) for name, other_embed in zip(dataset_names, embeddings) if name != dataset_name}
            argmax = max(dists.items(), key = lambda x: x[1])[0]
            res[dataset_name] = argmax
        return res
        
    dataset2closest = find_closest(dataset_names, embeddings, cosine_similarity)
    pretrain2downstream_results = get_pretrained_results(dataset_names, path_to_pretrained_logs)
    method_performance = {name: {'accuracy' : pretrain2downstream_results[dataset2closest[name]][name], 
                            'pretrain': dataset2closest[name]} for name in dataset_names}
    return method_performance

def get_random_baseline(dataset_names, path_to_pretrained_logs):
    pretrain2downstream_results = get_pretrained_results(dataset_names + ['imagenet'], path_to_pretrained_logs)
    random_performance = {}
    for name in dataset_names:
        if name == 'imagenet':
            continue
        choice = name
        while choice == name:
            choice = np.random.choice(dataset_names)
        random_performance[name] = {
            'accuracy': pretrain2downstream_results[choice][name],
            'pretrain': choice
        }
    return random_performance
            
def get_big_pretrain_baseline(dataset_names, path_to_pretrained_logs):
    pretrain2downstream_results = get_pretrained_results(dataset_names + ['imagenet'], path_to_pretrained_logs)
    big_baseline_performance = {}
    for name in dataset_names:
        if name == 'imagenet':
            continue
        big_baseline_performance[name] = {
            'accuracy': pretrain2downstream_results['imagenet'][name],
            'pretrain': 'imagenet'
        }
    return big_baseline_performance 

In [3]:
def cosine_similarity(e0, e1):
    return (e0*e1).sum()/np.linalg.norm(e0)/np.linalg.norm(e1)

def task2vec_resnet_embedder(dataset):
    resnet = get_model('resnet18', pretrained=True, num_classes=int(max(dataset.targets)+1)).cuda()
    task2vec_embedder = Task2Vec(resnet, skip_layers=6, max_samples=1000)
    return task2vec_embedder.embed(dataset)

dataset_names = ['mnist', 'cifar10', 'cifar100', 'letters', 'kmnist']
task2vec_res = get_embedder_results(
    dataset_names,
    path_to_pretrained_logs='/home/machenike/bmml/DataMetaMap/logs/pretrain_to_task_logs',
    embedder_func=task2vec_resnet_embedder,
    similarity_func=cosine_similarity
)

Caching features:   0%|          | 0/14 [00:00<?, ?it/s]

Fitting classifier:   0%|          | 0/10 [00:00<?, ?it/s]

Computing Fisher:   0%|          | 0/156 [00:00<?, ?it/s]

Caching features:   0%|          | 0/14 [00:00<?, ?it/s]

Fitting classifier:   0%|          | 0/10 [00:00<?, ?it/s]

...

Computing Fisher:   0%|          | 0/156 [00:00<?, ?it/s]

Caching features:   0%|          | 0/14 [00:00<?, ?it/s]

Fitting classifier:   0%|          | 0/10 [00:00<?, ?it/s]

Computing Fisher:   0%|          | 0/156 [00:00<?, ?it/s]

{'mnist': 'kmnist', 'cifar10': 'cifar100', 'cifar100': 'cifar10', 'letters': 'kmnist', 'kmnist': 'mnist'}


In [4]:
random_baseline_res = get_random_baseline(
    dataset_names,
    path_to_pretrained_logs='/home/machenike/bmml/DataMetaMap/logs/pretrain_to_task_logs',
)

In [5]:
imagenet_baseline_res = get_big_pretrain_baseline(
    dataset_names,
    path_to_pretrained_logs='/home/machenike/bmml/DataMetaMap/logs/pretrain_to_task_logs',
)

In [6]:
res = {
    'task2vec':{
        name:task2vec_res[name]['accuracy'] for name in dataset_names
    },
    'random':{
        name:random_baseline_res[name]['accuracy'] for name in dataset_names
    },
    'big_pretrain':{
        name:imagenet_baseline_res[name]['accuracy'] for name in dataset_names
    }
}


In [7]:
import pandas as pd

pd.DataFrame(res).T

,mnist,cifar10,cifar100,letters,kmnist
task2vec,0.9960,0.8798,0.4960,0.932596,0.9957
random,0.9815,0.6390,0.3512,0.933510,0.9752
big_pretrain,0.9848,0.8834,0.6789,0.919712,0.9851
